***Business task:*** Tier all active subscribers by churn risk, based on inactivity across billing, usage, and support.
Output feeds the retention team's daily outreach dashboard.

In [ ]:
# Cell 1 — Install
!apt-get install -y -qq mysql-server
!pip install -q pymysql cryptography ipython-sql sqlalchemy

In [ ]:
# Downgrade prettytable to a version known to work with ipython-sql
!pip install prettytable==3.10.2

# Alternatively, force output to Pandas to skip the prettytable formatter entirely
%config SqlMagic.autopandas = True

In [ ]:
# Cell 2 — Start MySQL + remove password
!service mysql start
!mysql --defaults-file=/etc/mysql/debian.cnf \
    -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY ''; FLUSH PRIVILEGES;"

In [ ]:
# Cell 3 — Upload file
from google.colab import files
files.upload()

In [ ]:
# Cell 4 — Load database (pure bash, one line)
!mysql -u root < telecom_complete.sql

In [ ]:
# Cell 5 — Connect Magic SQL
%load_ext sql
%sql mysql+pymysql://root@localhost/telecom
%config SqlMagic.displaylimit = 50

In [ ]:
%%sql
SHOW TABLES;

In [ ]:
%%sql

WITH
# The reference date represents the last date on the table..
ref_date AS (
  SELECT DATE('2024-03-25') AS today
),
last_bill_activity AS (
  SELECT
    subscriber_id,
    MAX(bill_date) AS last_activity
FROM bills
GROUP BY subscriber_id
),
last_usage_activity AS (
  SELECT
    s.subscriber_id,
    MAX(u.log_date) AS last_activity
FROM usage_logs  u
JOIN sims s USING(sim_id)
GROUP BY s.subscriber_id
),
last_ticket_activity AS (
  SELECT
    subscriber_id,
    MAX(opened_at) AS last_activity
  FROM support_tickets
  GROUP BY subscriber_id
),
all_activity AS (
  SELECT subscriber_id, last_activity FROM last_bill_activity   UNION ALL
  SELECT subscriber_id, last_activity FROM last_usage_activity  UNION ALL
  SELECT subscriber_id, last_activity FROM last_ticket_activity
),
last_seen AS (
  SELECT
    subscriber_id,
    MAX(last_activity) AS last_seen_date
  FROM all_activity
  GROUP BY subscriber_id
),
inactivity AS (
  SELECT
    ls.subscriber_id,
    ls.last_seen_date,
    DATEDIFF(rd.today, ls.last_seen_date) AS days_inactive,
    CASE
      WHEN DATEDIFF(rd.today, ls.last_seen_date) > 365 THEN 'Critical'
      WHEN DATEDIFF(rd.today, ls.last_seen_date) > 90  THEN 'High Risk'
      WHEN DATEDIFF(rd.today, ls.last_seen_date) > 30  THEN 'At Risk'
      WHEN DATEDIFF(rd.today, ls.last_seen_date) > 20  THEN 'Early Warning'
      ELSE                                                   'Healthy'
    END AS churn_risk
  FROM last_seen ls
  CROSS JOIN ref_date rd
),
billing_health AS (
  SELECT
    subscriber_id,
    ROUND(SUM(total_amount), 2) AS total_billed,
    ROUND(SUM(amount_paid),  2) AS total_paid,
    ROUND(SUM(total_amount - amount_paid), 2) AS outstanding,
    COUNT(CASE WHEN status IN ('Unpaid','Overdue')
    THEN 1 END) AS unpaid_count
  FROM bills
  GROUP BY subscriber_id
)
SELECT
  sub.subscriber_id,
  sub.full_name,
  sub.status,
  p.plan_name,
  p.monthly_fee,
  i.last_seen_date,
  i.days_inactive,
  i.churn_risk,
  COALESCE(bh.outstanding,   0) AS outstanding_balance,
  COALESCE(bh.unpaid_count,  0) AS unpaid_bills,
  RANK() OVER (
    ORDER BY p.monthly_fee DESC,
             i.days_inactive DESC
  ) AS revenue_risk_rank,
  CASE
    WHEN i.churn_risk = 'Critical'
      THEN 'Write-off review + win-back campaign'
    WHEN i.churn_risk = 'High Risk'
     AND COALESCE(bh.outstanding, 0) > 0
      THEN 'Urgent: debt collection + retention call'
    WHEN i.churn_risk IN ('At Risk','High Risk')
     AND p.monthly_fee >= 50
      THEN 'Priority call + loyalty discount offer'
    WHEN i.churn_risk = 'Early Warning'
      THEN 'Automated re-engagement email'
    ELSE 'Monitor — no action needed'
  END AS recommended_action
FROM subscribers sub
JOIN sims si USING(subscriber_id)
JOIN sim_plan_history sph ON si.sim_id = sph.sim_id
AND sph.end_date IS NULL
JOIN plans p ON sph.plan_id = p.plan_id
JOIN inactivity i ON sub.subscriber_id = i.subscriber_id
LEFT JOIN billing_health bh ON sub.subscriber_id = bh.subscriber_id
WHERE  i.churn_risk != 'Healthy'
ORDER  BY days_inactive DESC;